# Anatomía de una Falla · Parte II — 01 · Análisis exploratorio (EDA)

Este notebook recorre el dataset **NASA N-CMAPSS** antes de modelar. Cada sección explica **qué se mira y por qué**;
los números, tablas y gráficos se calculan desde los datos en `src/ncmapss/eda.py` (etapa `eda` de `scripts/run_all.py`)
y se leen de `outputs/<modo>/eda.json`. No hay valores escritos a mano: si cambian los datos o la configuración,
cambian las lecturas.

**Cómo regenerarlo:** `python scripts/run_all.py --only prepare health residuals features eda` (o `--sample` para la versión rápida).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import plotly.io as pio
pio.renderers.default = "notebook_connected"
from src.ncmapss import nb
MODE = "full"          # "sample" si corriste el pipeline con --sample

In [ ]:
# Opcional: recalcular el EDA desde los datos procesados (unos segundos)
# from src.ncmapss import eda; from src.ncmapss.config import load_config, get_paths
# eda.run(load_config(), get_paths(load_config(), MODE))
E = nb.load("eda", MODE)
print("EDA generado el", E["generated"], "· modo", E["mode"])
nb.facts()

## 1 · Estructura del dataset

N-CMAPSS tiene **un registro por segundo de vuelo**. Cada archivo `N-CMAPSS_DSxx.h5` es un *escenario de falla*
(qué componentes se degradan) y trae dos particiones: **desarrollo** (para entrenar) y **prueba** (motores nuevos).
Las variables vienen en seis bloques:

| bloque | contenido | ¿existe en un avión real? |
|---|---|---|
| `W` | condiciones de vuelo: altitud, Mach, potencia (TRA), temperatura de entrada (T2) | sí |
| `X_s` | 14 sensores físicos: temperaturas, presiones, giros, combustible | sí |
| `X_v` | 14 sensores virtuales del simulador | no |
| `T` | 10 parámetros de salud (eficiencia y flujo de cada componente) | no |
| `Y` | RUL: vuelos que le quedan al motor | es lo que se predice |
| `A` | motor, n° de vuelo, clase de vuelo (Fc) y estado de salud (hs) | sí (salvo hs) |

In [ ]:
nb.table('sizes')

In [ ]:
nb.show('sizes')

### Calidad de los datos: faltantes, infinitos y consistencia

Se revisan **todos los segundos** (no una muestra): nulos, infinitos, rangos físicos y reglas que el dataset debería
cumplir (RUL = vida total − n° de vuelo, vuelos numerados sin huecos, clase de vuelo fija por motor, estado de salud
que nunca vuelve a "sano"). También se chequea la integridad de cada archivo: uno de los `.h5` de Kaggle está truncado.

In [ ]:
nb.checks()

In [ ]:
nb.table('missing')

La **variación relativa** (desvío / media) muestra cuánto se mueve cada variable. Que varíe poco no significa que no sirva.

In [ ]:
nb.show('quality')

## 2 · Perfiles de vuelo y condiciones operativas

Cada ciclo es un **vuelo completo** con condiciones que cambian segundo a segundo. Los motores se agrupan en tres
**clases de vuelo**: corto (Fc = 1), mediano (Fc = 2) y largo (Fc = 3). Miramos un vuelo, la envolvente de operación,
la duración de cada clase y la forma típica de cada perfil.

In [ ]:
nb.show('profile')

In [ ]:
nb.show('envelope')

In [ ]:
nb.show('flight_class')
nb.table('flight_class')

In [ ]:
nb.show('flight_shapes')

## 3 · Contexto: separar el vuelo del desgaste

La pregunta clave: ¿cuánto de lo que mide un sensor depende de **cómo se vuela** y cuánto del **desgaste**?
Se entrena un *modelo de motor sano* (red neuronal chica) que predice cada sensor desde las 4 condiciones de vuelo,
usando sólo vuelos sanos de los motores de entrenamiento. Lo medido menos lo esperado es el **residuo**.

In [ ]:
nb.show('context')

In [ ]:
nb.show('variance_explained')

In [ ]:
nb.show('corr')

In [ ]:
nb.show('normalization')

### Los perfiles de vuelo se repiten

El simulador reutiliza un conjunto finito de **vuelos reales grabados**. Eso deja en el residuo un sesgo propio de
cada perfil, que conviene conocer antes de modelar (y que es central para la Línea 2: generalizar entre tipos de vuelo).

In [ ]:
nb.show('profiles')

## 4 · Qué componentes se desgastan

Los parámetros de salud (`T`) no se miden en la realidad, pero en la simulación muestran **qué se daña** en cada
escenario y **cómo evoluciona** el daño. El estado `hs` marca cuándo empieza la degradación anormal.

In [ ]:
nb.show('health_heat')
nb.table('health_heat')

In [ ]:
nb.show('health_traj')

In [ ]:
nb.show('hs')

## 5 · Evolución de los sensores y sensores más informativos

Ya sin el efecto del vuelo: qué sensores cambian al acercarse la falla (correlación de Spearman con la cercanía a la
falla), cómo evolucionan a lo largo de la vida, si hay valores atípicos y cómo se ve el estado del motor resumido en dos ejes.

In [ ]:
nb.show('rul_corr')

In [ ]:
nb.show('sensor_trends')

In [ ]:
nb.show('life_evolution')

In [ ]:
nb.show('outliers')
nb.table('outliers')

In [ ]:
nb.show('pca')

## 6 · Vida útil y distribución del RUL

In [ ]:
nb.show('lifespan')
nb.table('lifespan')

In [ ]:
nb.show('life_vs_fc')

In [ ]:
nb.show('rul_dist')

## 7 · Conclusiones del EDA → decisiones de modelado

Cada decisión del modelado (notebook `02_rul_incertidumbre.ipynb`) responde a un hallazgo de este análisis:

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
nb.decisions()

In [ ]:
pd.DataFrame(E['takeaways'])